In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [4]:

# Load dataset
data = np.load("processed_dataset.npz" , allow_pickle=True)
x_train, y_train = data['x_train'], data['y_train']
x_test, y_test = data['x_test'], data['y_test']

# Ensure data is numerical
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

# Encode labels
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# Convert labels to one-hot encoding
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)


In [5]:

# Model Architecture
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(x_train.shape[1], x_train.shape[2])),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Conv1D(filters=128, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(y_train.shape[1], activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=50,
    batch_size=32,
    verbose=1
)

# Save the model
model.save("trained_model.h5")

# Evaluate the model
eval_results = model.evaluate(x_test, y_test, verbose=1)
print(f"Test Accuracy: {eval_results[1] * 100:.2f}%")


Epoch 1/50
489/489 [==============================] - 4s 6ms/step - loss: 0.6708 - accuracy: 0.8186 - val_loss: 0.4204 - val_accuracy: 0.8811
Epoch 2/50
489/489 [==============================] - 3s 5ms/step - loss: 0.4313 - accuracy: 0.8764 - val_loss: 0.3355 - val_accuracy: 0.8990
Epoch 3/50
489/489 [==============================] - 3s 5ms/step - loss: 0.3890 - accuracy: 0.8841 - val_loss: 0.3407 - val_accuracy: 0.8987
Epoch 4/50
489/489 [==============================] - 3s 5ms/step - loss: 0.3645 - accuracy: 0.8909 - val_loss: 0.3058 - val_accuracy: 0.9010
Epoch 5/50
489/489 [==============================] - 2s 5ms/step - loss: 0.3287 - accuracy: 0.8981 - val_loss: 0.3223 - val_accuracy: 0.9008
Epoch 6/50
489/489 [==============================] - 3s 5ms/step - loss: 0.3111 - accuracy: 0.8994 - val_loss: 0.3400 - val_accuracy: 0.8993
Epoch 7/50
489/489 [==============================] - 3s 5ms/step - loss: 0.2979 - accuracy: 0.9072 - val_loss: 0.2316 - val_accuracy: 0.9243
Epoch 

In [6]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report



def evaluate_model(model, x_test, y_test, encoder=None):
    """
    Evaluate the model on test data and compute precision, recall, and F1-score.

    Parameters:
    - model: Trained model.
    - x_test: Test features.
    - y_test: True labels for test data (categorical or one-hot encoded).
    - encoder: LabelEncoder instance for decoding labels (optional).

    Returns:
    - Classification report with precision, recall, and F1-score.
    """
    # Ensure y_test is consistent
    if isinstance(y_test[0], str):
        if encoder is None:
            encoder = LabelEncoder()
            encoder.fit(y_test)  # Fit the encoder to the unique labels
        y_test = encoder.transform(y_test)

    # Get predictions
    y_pred = model.predict(x_test)
    y_pred_classes = np.argmax(y_pred, axis=1)

    # If y_test is one-hot encoded, decode it
    if len(y_test.shape) > 1 and y_test.shape[1] > 1:
        y_test = np.argmax(y_test, axis=1)

    # Compute metrics
    precision = precision_score(y_test, y_pred_classes, average='weighted')
    recall = recall_score(y_test, y_pred_classes, average='weighted')
    f1 = f1_score(y_test, y_pred_classes, average='weighted')

    # Print metrics
    print("Classification Report:")
    print(classification_report(y_test, y_pred_classes))
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")

   

    # return classification_report(y_test, y_pred_classes, output_dict=True)


# Evaluate the model
evaluate_model(model, x_test, y_test, encoder=encoder)


Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       201
           1       0.96      0.66      0.78       112
           2       0.98      0.96      0.97       734
           3       0.98      0.98      0.98      1010
           4       0.89      0.56      0.69       128
           5       0.91      0.99      0.95       961
           6       0.97      0.99      0.98       765

    accuracy                           0.96      3911
   macro avg       0.95      0.88      0.91      3911
weighted avg       0.96      0.96      0.96      3911

Precision: 0.96
Recall: 0.96
F1 Score: 0.96
